# 1. AMES HOUSING - Preprocesamiento
**Tipo:** REGRESIÓN (predecir precio)
**Variable objetivo:** SalePrice

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# CARGAR DATOS (Excel)
df = pd.read_excel('AmesHousing.xls')
print(f"Dimensiones: {df.shape}")
print(f"Columnas: {list(df.columns[:10])}...")  # Primeras 10

In [ ]:
# ============================================================
# LIMPIEZA DE NULOS
# ============================================================
# En Ames, NaN en Pool, Fence, Alley significa "No tiene", NO es error

# Columnas donde NaN = "No tiene" (rellenar con 'None' o 0)
cols_none = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']

for col in cols_none:
    if col in df.columns:
        df[col] = df[col].fillna('None')

# Columnas numéricas con NaN = 0
cols_zero = ['GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2',
             'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']

for col in cols_zero:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Resto de nulos: rellenar con moda (categóricas) o mediana (numéricas)
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())

print(f"Nulos restantes: {df.isnull().sum().sum()}")

In [ ]:
# ============================================================
# SEPARAR VARIABLE OBJETIVO
# ============================================================
y = df['SalePrice'].values

# Eliminar columnas no útiles
cols_drop = ['Order', 'PID', 'SalePrice']  # IDs y objetivo
X_df = df.drop(columns=[c for c in cols_drop if c in df.columns])

print(f"Variable objetivo (y): {y.shape}")
print(f"Características: {X_df.shape}")

In [ ]:
# ============================================================
# ENCODING DE VARIABLES CATEGÓRICAS
# ============================================================
# Identificar columnas categóricas y numéricas
cat_cols = X_df.select_dtypes(include=['object']).columns.tolist()
num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()

print(f"Categóricas: {len(cat_cols)}")
print(f"Numéricas: {len(num_cols)}")

# One-Hot Encoding para categóricas
X_encoded = pd.get_dummies(X_df, columns=cat_cols, drop_first=True)
print(f"\nDimensiones después de encoding: {X_encoded.shape}")

In [ ]:
# ============================================================
# DIVISIÓN DE DATOS (70/20/10)
# ============================================================
X = X_encoded.to_numpy()

# Primera división: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)

# Segunda división: 20% val, 10% test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.333, random_state=42)

print(f"Train: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Val:   {X_val.shape[0]} ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test:  {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")

In [ ]:
# ============================================================
# NORMALIZACIÓN (después de dividir para evitar data leakage)
# ============================================================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)  # fit solo en train
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("✅ Datos listos para modelo de REGRESIÓN")
print(f"Características: {X_train.shape[1]}")

## MÉTODO TRADICIONAL (Sin Redes Neuronales)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error

# Regresión Lineal tradicional
modelo_lr = LinearRegression()
modelo_lr.fit(X_train, y_train)

y_pred = modelo_lr.predict(X_test)
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")

## CON PYTORCH (Red Neuronal)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Convertir a tensores
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).reshape(-1, 1)
X_val_t = torch.FloatTensor(X_val)
y_val_t = torch.FloatTensor(y_val).reshape(-1, 1)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).reshape(-1, 1)

# DataLoaders
train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

print(f"Input features: {X_train_t.shape[1]}")

In [ ]:
# Modelo de regresión con PyTorch
class RegresionNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)  # Salida: 1 valor (precio)
        )
    
    def forward(self, x):
        return self.red(x)

modelo = RegresionNN(X_train_t.shape[1])
criterio = nn.MSELoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.001)
print(modelo)